<a href="https://colab.research.google.com/github/sbleeedu1-ai/python_CLI_board_example_26_08/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_CLI_board.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 게시판
from datetime import datetime
from pytz import timezone
from dataclasses import dataclass
import getpass    # 비밀번호 보이지 않게

@dataclass
class Article:
  id: int
  regDate: str
  updateDate: str
  title: str
  body: str

@dataclass
class Member:
  id: str
  password: str

# 생성 시 회원에게 필요한 변수들 선언


def now():
  return datetime.now(timezone('Asia/Seoul')).strftime("%Y-%m-%d %H:%M:%S")

def is_cmd_right(cmd):
  cmd_bits = cmd.split(" ")

  if len(cmd_bits) < 3:
    print("명령어를 다시 입력해주세요 (글 번호 없음)")
    return None

  if not cmd_bits[2].isdigit():
    print("명령어를 다시 입력해주세요 (번호대신 문자입력)")
    return None

  return int(cmd_bits[2])

def is_exist(lists,lists_id):
  for obj in lists:
    if obj.id == lists_id:
      return obj
  return None

def format_date(regDate):
  today = now().split(" ")[0]
  write_date = regDate.split(" ")[0]
  write_time = regDate.split(" ")[1]
  if today == write_date:
    return write_time
  else:
    return write_date

def makeTestData():
  return [
      Article(1, "2025-12-12 12:12:12", "2025-12-12 12:12:12", "제목1", "내용1"),
      Article(2, now(), now(), "제목2", "내용2"),
      Article(3, now(), now(), "제목3", "내용3")
          ]

def pw_check(pw):
  for c in pw:
    if c.isupper():
      if pw.isalnum():
        return True
      else:
        print("유효하지 않은 비밀번호 입니다.(특수문자 제외)")
        return False

  print("유효하지 않은 비밀번호 입니다.(대문자 포함 필수)")
  return False

# isupper 를 잘못 알았다. 모든 글자가 대문자일때만 True
# 한글자씩 따와서 확인
# 함수로 구현?

def is_pw_right(lists,mem_id,mem_pw):
  for obj in lists:
    if obj.id == mem_id:
      if obj.password == mem_pw:
        return obj
  return None

# is_exist 함수와 동일하게 리스트에서 꺼내 하나씩 대조
# 대조 항목이 추가되어 분리
# 맞는 아이디인 경우에만 그 객체의 암호를 체크

print("== CLI 게시판 실행 ==")
article_num = 3
articles = makeTestData()

members=[]       # 가입한 회원들
loggedIn=[]      # 로그인 상태 저장

while True:
  user_cmd = input("명령어 ) ").strip()

  if user_cmd == 'exit':
    break

  elif user_cmd == 'article write':
    article_num += 1
    title = input("제목 : ")
    content = input("내용 : ")
    article = Article(article_num,now(),now(),title,content)
    articles.append(article)
    print(f"{article_num}번 글이 생성되었습니다.")

  elif user_cmd == 'article list':
    if not articles:
      print("작성한 글이 없습니다.")
    else:
      print("========================================================")
      print("번호".ljust(5),end='/')
      print("    제목".ljust(10),end='/')
      print("    내용".ljust(10),end='/')
      print("    작성시간".ljust(10))
      for a in reversed(articles):
          print(f"  {a.id}".ljust(8),end='/')
          print(f"     {a.title}".ljust(10),end='/')
          print(f"     {a.content}".ljust(10),end='/')
          print(f"     {format_date(a.regDate)}")
      print("========================================================")

  elif user_cmd.startswith('article delete'):
    deletedId = is_cmd_right(user_cmd)
    if deletedId is None:
      continue

    article = is_exist(articles, deletedId)
    if article is None:
      print(f"{deletedId}번 글은 존재하지 않습니다.")
      continue

    articles.remove(article)
    print(f"{deletedId}번 글이 삭제 되었습니다.")

  elif user_cmd.startswith('article edit'):
    editedId = is_cmd_right(user_cmd)
    if editedId is None:
      continue

    article = is_exist(articles, editedId)
    if article is None:
      print(f"{editedId}번 글은 존재하지 않습니다.")
      continue

    print(f"기존 제목 : {article.title}")
    print(f"기존 내용 : {article.content}")
    article.title = input("새 제목 : ")
    article.content = input("새 내용 : ")
    article.updateDate = now()
    print(f"{editedId}번 글이 수정 되었습니다.")

  elif user_cmd.startswith('article detail'):
    detailId = is_cmd_right(user_cmd)
    if detailId is None:
      continue

    article = is_exist(articles, detailId)
    if article is None:
      print(f"{detailId}번 글은 존재하지 않습니다.")
      continue

    print(f"번호 : {article.id}")
    print(f"작성 날짜 : {article.regDate}")
    print(f"수정 날짜 : {article.updateDate}")
    print(f"제목 : {article.title}")
    print(f"내용 : {article.content}")

  elif  user_cmd == 'member join':
    # member_count += 1
    member_id = input("아이디 : ")
    memberId = is_exist(members, member_id)
    if memberId :
      print("이미 존재하는 아이디 입니다.")
      continue

    member_pw = getpass.getpass("암호 : ")
    memberPw = pw_check(member_pw)
    if not memberPw:
      continue

    member = Member(member_id,member_pw)
    members.append(member)
    print("회원 가입을 축하합니다.")
    print(members)

  # 회원 Member
  # 회원 가입
  # member join
  # - 아이디 중복체크
  # -> 없으면 다시 명령어
  # 중복 체크를 통과한 경우에만 -> 비밀번호 입력
  # 비밀번호는 가려져 보이게 getpass 사용
  # - 비밀번호 유효성 체크
  # 대문자 1개 포함, 문자숫자 자유, 특수문자 제한


  elif  user_cmd == 'member login':
    login_id = input("아이디 : ")
    login_pw = getpass.getpass("암호 : ")

    loginId = is_exist(members, login_id)
    loginPw = is_pw_right(members, login_id, login_pw)
    # print(loginId,loginPw)
    # 틀린 암호 문구가 계속 나와 확인
    # 까보니 함수 만들때 객체 리턴 시켰고 요소로 그 객체를 전달함
    # 막상 체크는 입력한 암호를 했어야 함
    if loginId is None:
      print("존재하지 않는 아이디 입니다.")
      continue
    if loginPw is None:
      print("비밀번호가 틀렸습니다.")
      continue

    loggedIn.append(loginId)
    print("로그인 성공")

  # 로그인
  # member login
  # 로그인 시 아이디 암호를 동시에 받아야 하나
  # 아이디 존재여부 체크
  # 암호는 맞는 아이디의 암호만을 체크해야함
  # 다 통과 시 로그인 리스트에 저장


  elif user_cmd.startswith('member logout'):
    logged_in_Id = is_cmd_right(user_cmd)
    if detailId is None:
      continue


  # 로그아웃
  # member logout
  # 로그인된 계정을
  # -> 이걸 어떻게 구분하지?
  # 로그인 리스트에 있으면?
  # 로그아웃 할 계정을 뒤에 입력시키자
  # 없다면 로그인 중 아니다 출력
  # 있다면 로그아웃 처리 (로그인 중인 리스트에서 빼기)



  # 회원이 아니면 article write, delete, edit 못하게?
  # 근데 article list / detail 은 가능?
  # 그리고 로그인 상태일 때 write 시 회원 정보도 기억?
  # '내'가 로그인 상태를 write 시 어떻게 구분시키지?
  # 이러면 로그인 중 리스트에는 1개만 들어가게?

  else:
    print("지원하지 않은 명령어 입니다.")

print("== CLI 게시판 종료 ==")


== CLI 게시판 실행 ==


KeyboardInterrupt: Interrupted by user